# Run func-e

First we need to encode our target reaction.

To enable us doing this we needed to change rxnfp because it has odd deps so can't be easily installed.


In [3]:
! pip install unimol-tools

  Using cached unimol_tools-0.1.5-py3-none-any.whl.metadata (11 kB)
  Using cached addict-2.4.0-py3-none-any.whl.metadata (1.0 kB)
  Using cached numba-0.65.1-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (2.9 kB)
  Using cached hydra_core-1.3.2-py3-none-any.whl.metadata (5.5 kB)
  Using cached omegaconf-2.3.0-py3-none-any.whl.metadata (3.9 kB)
  Using cached tensorboard-2.20.0-py3-none-any.whl.metadata (1.8 kB)
  Using cached antlr4_python3_runtime-4.9.3-py3-none-any.whl
  Using cached llvmlite-0.47.0-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (5.0 kB)
  Using cached absl_py-2.4.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached grpcio-1.80.0-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (3.8 kB)
  Using cached markdown-3.10.2-py3-none-any.whl.metadata (5.1 kB)
  Using cached tensorboard_data_server-0.7.2-py3-none-manylinux_2_31_x86_64.whl.metadata (1.1 kB)
  Using cached werkzeug-3.1.8-py3-none-any.whl.metadata (4.

In [1]:
# Imports 
import pandas as pd
from ast import literal_eval
import numpy as np
import os 
import sys
from enzymetk.sequence_search_blast import BLAST
from enzymetk.similarity_foldseek_step import FoldSeek
from enzymetk.similarity_reaction_step import ReactionDist
from enzymetk.save_step import Save
import pandas as pd
import os
os.environ['MKL_THREADING_LAYER'] = 'GNU'
import pandas as pd
import sys
# sys.path.append('/disk1/ariane/vscode/enzyme-tk/')
from enzymetk.embedchem_unimol_step import UniMol
from enzymetk.embedprotein_esm_step import EmbedESM
from enzymetk.save_step import Save
from enzymetk.embedchem_drfp_step import DRFP

# Given the reaction can't be fully described with the side chains this one is risky...
reaction = 'CCCCC(CC)COC(=O)C1=CC=CC=C1C(=O)OCC(CC)CCCC>>CCCCC(CC)COC(=O)C1=CC=CC=C1C(=O)O' # Note the substrate can be passed but here we just calculate everything
name = 'DEHP-MEHP'
sequence_file = ''

Boltz: Needs docko package.Install with: pip install docko. Error: {e}
Chai: Needs docko package. Install with: pip install docko. Error: No module named 'docko'
Vina: Needs docko package. Install with: pip install docko. Error: {e}


/mnt/storage01/home/amora/.conda/envs/enzymetk/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


EmbedESM3: Needs esm3 package. Install with: pip install esm.


In [4]:
substrate = reaction.split('>>')[0]
product = reaction.split('>>')[-1]
df = pd.DataFrame([[name, substrate, product, reaction]], columns=['name', 'substrate', 'product', 'reaction'])
# Saves the data in pickle format (not otherwise the representations aren't saved correctly!)
df = df << (DRFP(smiles_col='reaction'))
df = df << (UniMol(smiles_col='substrate'))
df = df.rename(columns={'unimol_repr': 'substrate_unimol'})
df = df << (UniMol(smiles_col='product'))
df = df.rename(columns={'unimol_repr': 'product_unimol'})
# Save 
df.to_pickle(f'examples_output/user_reaction_{name}.pkl')

Encoding SM
2026-05-25 14:38:57 | unimol_tools/weights/weighthub.py | 54 | INFO | Uni-Mol Tools | Weights will be downloaded to default directory: /mnt/storage01/home/amora/.conda/envs/enzymetk/lib/python3.10/site-packages/unimol_tools/weights
2026-05-25 14:38:57 | unimol_tools/weights/weighthub.py | 95 | INFO | Uni-Mol Tools | Downloading modelzoo/164M/checkpoint.pt


['Failed to encode 0 SMILES strings. Successfully encoded 1 SMILES strings.']


Fetching 1 
2026-05-25 14:39:04 | unimol_tools/models/unimolv2.py | 176 | INFO | Uni-Mol Tools | Loading pretrained weights from /mnt/storage01/home/amora/.conda/envs/enzymetk/lib/python3.10/site-packages/unimol_tools/weights/modelzoo/164M/checkpoint.pt
2026-05-25 14:39:04 | unimol_tools/data/conformer.py | 452 | INFO | Uni-Mol Tools | Start generating conformers...
1it [00:00,  1.88it/s]
2026-05-25 14:39:05 | unimol_tools/data/conformer.py | 467 | INFO | Uni-Mol Tools | Succeeded in generating conformers for 100.00% of molecules.
2026-05-25 14:39:05 | unimol_tools/data/conformer.py | 484 | INFO | Uni-Mol Tools | Succeeded in generating 3d conformers for 100.00% of molecules.
2026-05-25 14:39:05 | unimol_tools/tasks/trainer.py | 103 | INFO | Uni-Mol Tools | Using CPU.
100%|█| 1/1
2026-05-25 14:39:06 | unimol_tools/models/unimolv2.py | 176 | INFO | Uni-Mol Tools | Loading pretrained weights from /mnt/storage01/home/amora/.conda/envs/enzymetk/lib/python3.10/site-packages/unimol_tools/wei

# Now we load and run func-e

In [1]:
import pickle
from colab_ml_train import *
    
def average_retransformed_features(df_to_add, dfs, enzyme_cols, reaction_cols, label=''):
    for enzyme_col in enzyme_cols + reaction_cols:
        data = []
        for d in dfs:
            data.append(d[f'inverse_transformed_pred_{enzyme_col}'].values)
        data = np.array(data)
        df_to_add[f'{label}{enzyme_col}_mean'] = np.mean(data, axis=0)
        df_to_add[f'{label}{enzyme_col}_std'] = np.std(data, axis=0)

    return df_to_add
    
    
def retransform_scaled_predictions(test, pred, enzyme_feature_scaler, enzyme_cols, reaction_feature_scaler, reaction_cols):
    # Test how the test set accuracy is, also check what the length looks like for this prediction!
    pred_enzyme_cols = []
    for i, v in enumerate(enzyme_cols):
        test[f'pred_{v}'] = pred[:, i+1].detach()
        pred_enzyme_cols.append(f'pred_{v}')
        
    scaled_cols = enzyme_feature_scaler.inverse_transform(test[pred_enzyme_cols].values)
    for i, v in enumerate(pred_enzyme_cols):
        test[f'inverse_transformed_{v}'] = scaled_cols[:, i]
    
    pred_reaction_cols = []
    # Do the same for the reactions
    for i, v in enumerate(reaction_cols):
        test[f'pred_{v}'] = pred[:, i+1+len(pred_enzyme_cols)].detach()
        pred_reaction_cols.append(f'pred_{v}')
        
    scaled_cols = reaction_feature_scaler.inverse_transform(test[pred_reaction_cols].values)
    for i, v in enumerate(pred_reaction_cols):
        test[f'inverse_transformed_{v}'] = scaled_cols[:, i]
        
    test['pred_Activity'] = pred[:, 0].detach()
    return test
    
def predict_on_dataset_ensemble(label, df, models, protein_embedding_column, product_embedding_column, 
                                substrate_embedding_column, reaction_embedding_column, enzyme_feature_scaler, 
                                reaction_feature_scaler,
                                action_column=None, seq_column=None, plot_fig=False):
    enzyme_cols = ['Length', 'Mass', 'Polarity', 'temperature']
    reaction_cols = ['substrates_MolWt', 'substrates_MolLogP', 'substrates_MaxPartialCharge', 'substrates_MinPartialCharge', 
                 'products_MolWt', 'products_TPSA', 'products_MolLogP', 'products_MaxPartialCharge', 'products_MinPartialCharge']
    
    if not action_column:
        action_column = 'EmptyAction'
        df[action_column] = 0
    y_true = df[action_column].values

    torch.set_float32_matmul_precision('high')
    torch.set_default_dtype(torch.float32)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Collect predictions from all models
    all_model_preds = []
    validation_dfs = []
    with torch.no_grad():
        with torch.cuda.amp.autocast(enabled=False):
            for model in models:
                model.eval()
                for module in model.modules():
                    if isinstance(module, (torch.nn.BatchNorm1d, torch.nn.BatchNorm2d, torch.nn.BatchNorm3d)):
                        module.eval()
                        module.track_running_stats = False
                        module.running_mean = module.running_mean.detach()
                        module.running_var = module.running_var.detach()
                model.to(device)
                model.eval()

                X_batch_enzyme = torch.tensor(df[protein_embedding_column].values.tolist()).to(device)
                X_batch_product = torch.tensor(df[product_embedding_column].values.tolist()).to(device)
                X_batch_substrate = torch.tensor(df[substrate_embedding_column].values.tolist()).to(device)
                X_batch_reaction = torch.tensor(df[reaction_embedding_column].values.tolist()).to(device)

                pred = model(X_batch_enzyme, X_batch_product, X_batch_substrate, X_batch_reaction)

                # Sigmoid since do this in the loss normally
                pred = pred.cpu()
                probs = torch.sigmoid(pred[:, 0].squeeze())
                pred[:, 0] = probs.float()

                validation_df = retransform_scaled_predictions(df, pred, enzyme_feature_scaler, enzyme_cols, 
                                                       reaction_feature_scaler, reaction_cols)
                all_model_preds.append(pred.numpy())
                validation_dfs.append(validation_df.copy())

    all_model_preds = np.stack(all_model_preds, axis=0)
    print('ALL', all_model_preds.shape)

    # Calc mean and variance 
    mean_preds = np.mean(all_model_preds, axis=0)
    std_preds = np.std(all_model_preds, axis=0)

    # Classification metrics
    y_pred = np.round(mean_preds[:, 0])
    acc = np.mean(y_pred == y_true)
    f1 = f1_score(y_true, y_pred, average='macro')
    auprc = average_precision_score(y_true, y_pred)

    print(f"Accuracy: {acc:.4f}, F1: {f1:.4f}, AUPRC: {auprc}")
    return mean_preds, std_preds, f1, acc, validation_dfs


def run_funce(validation_df, model_dir, model_labels, reaction, action_column, protein_embedding_column, 
              product_embedding_column, substrate_embedding_column, reaction_embedding_column):
    models = []
    for model_label in model_labels:
        if os.path.exists(f'{model_dir}/{model_label}_conf.pkl'):
            model, config, optimizer = load(f'{model_dir}/{model_label}_conf.pkl', f'{model_dir}/{model_label}_checkpoint.pth')
            enzyme_feature_scaler = config['enzyme_feature_scaler']
            reaction_feature_scaler = config['reaction_feature_scaler']
            print(config)
            models.append(model)
        else:
            print(f'Model: {model_dir}/{model_labels}_conf.pkl DID NOT EXIST. Check your path or model label.')
            continue
    
    enzyme_cols = ['Length', 'Mass', 'Polarity', 'temperature']
    reaction_cols = ['substrates_MolWt', 'substrates_MolLogP', 'substrates_MaxPartialCharge', 'substrates_MinPartialCharge', 
                 'products_MolWt', 'products_TPSA', 'products_MolLogP', 'products_MaxPartialCharge', 'products_MinPartialCharge']
    
    
    mean_pred, std_preds, f1, acc, validation_dfs = predict_on_dataset_ensemble(action_column, validation_df, models, protein_embedding_column, product_embedding_column, 
                                                                                      substrate_embedding_column, reaction_embedding_column, 
                                                                                      enzyme_feature_scaler, reaction_feature_scaler, action_column, seq_column='Sequence')
    validation_df[f'{reaction}_prediction'] = mean_pred[:, 0]
    validation_df[f'{reaction}_std_preds'] = std_preds[:, 0]
    validation_df = average_retransformed_features(validation_df, validation_dfs, enzyme_cols, reaction_cols, f'{reaction}_')
    # Save dataframe to go through later if needed
    validation_df = validation_df.sort_values(by=f'{reaction}_prediction', ascending=False)
    return validation_df


/mnt/storage01/home/amora/.conda/envs/enzymetk/lib/python3.10/site-packages/torch/cuda/__init__.py:187: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


In [5]:
import pickle as pkl

with open(f'retraining/input_data/embeddings_drfp.pkl', 'rb') as fin:
    embeddings = pkl.load(fin)
    reaction_to_embedding = embeddings['reaction_to_drfp']
    reaction_to_substrate = embeddings['reaction_to_substrate_unimol']
    reaction_to_product = embeddings['reaction_to_product_unimol']
    uniprot_to_esm3 = embeddings['uniprot_to_esm2']
    


In [8]:
reaction_level = 'medium'
reaction_test_df = pd.read_csv(f'../CARE/splits/task2/{reaction_level}_reaction_test.csv')

In [9]:
protein_all_df = pd.read_csv(f'../CARE/splits/task1/protein_train.csv')
# Include in all for one example to test how the performance changes
test_ecs = set(reaction_test_df['EC number'].values)
protein_test_df = protein_all_df[protein_all_df['EC number'].isin(test_ecs)]
protein_test_df

,Unnamed: 0,Entry,Entry Name,Sequence,EC number,Length,EC All,clusterRes30,clusterRes50,clusterRes70,clusterRes90,EC3,EC2,EC1
60,65,A0A0H2ZMF9,PBP2A_STRP2,MKLDKLFEKFLSLFKKETSELEDSDSTILRRSRSDRKKLAQVGPIR...,3.4.16.4,731,3.4.16.4,A0A0H2ZMF9,A0A0H2ZMF9,A0A0H2ZMF9,A0A0H2ZMF9,3.4.16,3.4,3
65,70,A0A0H3GGY3,PGPH_LISM4,MKLAKKWRDWYIESGKKYLFPLLLVCFAVIAYFLVCQMTKPESYNV...,3.1.4.59,718,3.1.4.59,A0A0H3GGY3,A0A0H3GGY3,A0A0H3GGY3,A0A0H3GGY3,3.1.4,3.1,3
96,101,A0A0X1KHF9,RPH_LISMF,MKPYVLKFQEIRPHSEALVGGKGMNLGACSNIEGVHVPAGFCLTTE...,2.7.9.6,867,2.7.9.6,Q81BR3,Q81BR3,A0A0X1KHF9,A0A0X1KHF9,2.7.9,2.7,2
130,135,A0A1D6LAB7,RH3B_MAIZE,MASLTLPALALALSNPGAVRLRAAAFRCWALRRRGWAAAGALASPN...,3.6.4.13,743,3.6.4.13,Q9LUW5,Q0DM51,A0A1D6LAB7,A0A1D6LAB7,3.6.4,3.6,3
140,145,A0A1D8PPK1,EBP1_CANAL,MTIESTNSFVVPSDTELIDVTPLGSTKLFQPIKVGNNVLPQRIAYV...,1.6.99.1,407,1.6.99.1,Q09671,P43084,P43084,P43084,1.6.99,1.6,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
184478,185943,Q9ZMJ9,HCPE_HELPJ,MNIKILKILVGGLFFLSLNAHLWGKQDNSFLGIGERAYKSGNYSKA...,3.5.2.6,355,3.5.2.6,Q9ZMJ9,Q9ZMJ9,Q9ZMJ9,Q9ZMJ9,3.5.2,3.5,3
184481,185946,Q9ZMQ7,Y161_HELPJ,MKKNILNLALVGALSASFLMAKPAHNANNSTHNTKETTDASAGVLA...,5.2.1.8,299,5.2.1.8,Q0PAS1,Q9ZMQ7,Q9ZMQ7,Q9ZMQ7,5.2.1,5.2,5
184482,185947,Q9ZMU7,SDHL_HELPJ,MASFSILSIFKIGVGPSSSHTIGPMEAGARFCGLLKGILEQVERVQ...,4.3.1.17,455,4.3.1.17,O86564,Q9ZMU7,Q9ZMU7,Q9ZMU7,4.3.1,4.3,4
184489,185954,Q9ZNA8,TNAA_PROIN,MAKRIVEPFRIKMVENIRIPSREEREVALKEAGYNPFLLPSSAVYI...,4.1.99.1,465,4.1.99.1,Q9YCI2,P31015,Q9ZNA8,Q9ZNA8,4.1.99,4.1,4


In [10]:
reaction_embedding_column = 'rxnfp'
product_embedding_column = 'product_unimol_repr'
substrate_embedding_column = 'substrate_unimol_repr'
protein_embedding_column = 'esm3_mean'
reaction_test_df['rxnfp'] = [reaction_to_embedding.get(r) for r in reaction_test_df['Reaction'].values]
reaction_test_df['substrate_unimol_repr'] = [reaction_to_substrate.get(r) for r in reaction_test_df['Reaction'].values]
reaction_test_df['product_unimol_repr'] = [reaction_to_product.get(r) for r in reaction_test_df['Reaction'].values]
reaction_test_df[reaction_embedding_column] = [np.array(x).flatten().astype(np.float32) for x in reaction_test_df[reaction_embedding_column].values]
reaction_test_df[product_embedding_column] = [np.array(x).flatten().astype(np.float32)  for x in reaction_test_df[product_embedding_column].values]
reaction_test_df[substrate_embedding_column] = [np.array(x).flatten().astype(np.float32)  for x in reaction_test_df[substrate_embedding_column].values]

In [12]:
prediction_rows = []
reaction_test_df['id'] = [f'r{eid}_{ec}' for eid, ec in enumerate(reaction_test_df['EC number'].values)]

In [11]:
! mkdir predictions

In [16]:
from sciutil import SciUtil
import torch

def load(config_path, checkpoint_path):
    checkpoint = torch.load(checkpoint_path, weights_only=False, map_location=torch.device('cpu'))
    model = checkpoint['model']
    optimizer = checkpoint['optimizer']
    config = pickle.load(open(config_path, 'rb'))
    return model, config, optimizer
    
all_validations = pd.DataFrame()
u = SciUtil()
protein_level = '30'
k = 100
validation_df = protein_test_df.copy()
validation_df['esm3_mean'] = [uniprot_to_esm3.get(e) for e in protein_test_df['Entry'].values]
validation_df = validation_df.dropna(subset='esm3_mean')
validation_df['esm3_mean'] = [np.array(e).flatten().astype(np.float32) for e in validation_df['esm3_mean'].values]
validation_df = validation_df.drop_duplicates(subset='Entry')
validation_df = validation_df.dropna(subset=['esm3_mean'])
        

for i, row in reaction_test_df.iterrows():
    if not os.path.exists(f'predictions/{row["id"]}_newMLModels.csv'):
        true_ec = row['EC number']
        # Now for each of these reactions we would evaluate against all proteins
        validation_df[reaction_embedding_column] = [row[reaction_embedding_column] for i in range(len(validation_df))]
        validation_df[product_embedding_column] = [row[product_embedding_column] for i in range(len(validation_df))]
        validation_df[substrate_embedding_column] = [row[substrate_embedding_column] for i in range(len(validation_df))]
        validation_df['Action'] = [0 if ec != true_ec else 1 for ec in validation_df['EC number'].values]

        action_column = 'Action'
        missing_data = []
        
        # Settings for the model 
        model_type =  'ESRP'
        num_pairs = 500000
        label = 'colab'
        models = []
        model_dir = f'../../trained_models_colab_25052026/' #model_dir #/disk1/ariane/vscode/cec_degrader/publication_01202025/Archive/models_publication_08092025
        rows = []
        missing_data = []
        validation_df_dict = {}
        
        # Change the name here i
        reaction = 'rxn_'
        
        models = []
        model_idx = 1
        for ec_level in range(1, 5):
            if os.path.exists(f'{model_dir}/{label}_{reaction_level}_{protein_level}_{ec_level}_model_{model_idx}_{num_pairs}_conf.pkl'):
                model, config, optimizer = load(f'{model_dir}/{label}_{reaction_level}_{protein_level}_{ec_level}_model_{model_idx}_{num_pairs}_conf.pkl', 
                                                f'{model_dir}/{label}_{reaction_level}_{protein_level}_{ec_level}_model_{model_idx}_{num_pairs}_checkpoint.pth')
                
                enzyme_feature_scaler = config['enzyme_feature_scaler']
                reaction_feature_scaler = config['reaction_feature_scaler']
                print(config)
                models.append(model)
            else:
                print('DID NOT EXIST')
                print(f'{model_dir}/{label}_{reaction_level}_{protein_level}_{model_type}_{ec_level}_model_{model_idx}_{num_pairs}_conf.pkl')
                continue
        
        enzyme_cols = ['Length', 'Mass', 'Polarity', 'temperature']
        reaction_cols = ['substrates_MolWt', 'substrates_MolLogP', 'substrates_MaxPartialCharge', 'substrates_MinPartialCharge', 
                     'products_MolWt', 'products_TPSA', 'products_MolLogP', 'products_MaxPartialCharge', 'products_MinPartialCharge']
        
        
        mean_pred, std_preds, f1, acc, validation_dfs = predict_on_dataset_ensemble(action_column, validation_df, models, protein_embedding_column, product_embedding_column, 
                                                                                          substrate_embedding_column, reaction_embedding_column, 
                                                                                          enzyme_feature_scaler, reaction_feature_scaler, action_column, seq_column='Sequence')
        validation_df[f'{reaction}_prediction'] = mean_pred[:, 0]
        validation_df[f'{reaction}_std_preds'] = std_preds[:, 0]
        validation_df[f'{reaction}_epistemic'] = True #epistemic
        validation_df = average_retransformed_features(validation_df, validation_dfs, enzyme_cols, reaction_cols, f'rxn_')
        validation_df['Reaction'] = row['Reaction']
        validation_df['True EC'] = true_ec
        validation_df['members EC'] = row['members EC']
        validation_df['Reaction Text'] = row['Reaction Text']
        validation_df['Reaction id'] = row['id']
        # Save dataframe to go through later if needed
        validation_df = validation_df.sort_values(by=f'{reaction}_prediction', ascending=False)
        validation_df[[c for c in validation_df.columns if c not in [reaction_embedding_column, product_embedding_column, 
                                                      substrate_embedding_column, 'esm3_mean']]].to_csv(f'predictions/{row["id"]}_newMLmodels_{reaction_level}.csv')
        top_k_pred = validation_df.sort_values(by=f'{reaction}_prediction', ascending=False)['EC number'].values[:k]
        prediction_rows.append(list(row) + [top_k_pred])
# colab_easy_30_1_model_1_5000000_conf


{'enzyme_feature_scaler': MinMaxScaler(), 'reaction_feature_scaler': MinMaxScaler(), 'config': {'layers': [512, 256, 128, 0, 0], 'dropout': 0.05, 'num_epochs': 20, 'batch_size': 1000, 'output_dim': 14, 'learning_rate': 0.0001, 'num_heads': 8, 'early_stop': 5, 'attention': {'product_size': 768, 'substrate_size': 768, 'reaction_size': 2048, 'enzyme_size': 2560, 'embed_size': 1024, 'attention_type': 'cross'}}}
{'enzyme_feature_scaler': MinMaxScaler(), 'reaction_feature_scaler': MinMaxScaler(), 'config': {'layers': [512, 256, 128, 0, 0], 'dropout': 0.05, 'num_epochs': 20, 'batch_size': 1000, 'output_dim': 14, 'learning_rate': 0.0001, 'num_heads': 8, 'early_stop': 5, 'attention': {'product_size': 768, 'substrate_size': 768, 'reaction_size': 2048, 'enzyme_size': 2560, 'embed_size': 1024, 'attention_type': 'cross'}}}
{'enzyme_feature_scaler': MinMaxScaler(), 'reaction_feature_scaler': MinMaxScaler(), 'config': {'layers': [512, 256, 128, 0, 0], 'dropout': 0.05, 'num_epochs': 20, 'batch_size': 

/tmp/ipykernel_4159240/1202899340.py:61: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):
/tmp/ipykernel_4159240/1202899340.py:73: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  X_batch_enzyme = torch.tensor(df[protein_embedding_column].values.tolist()).to(device)


ALL (4, 12081, 14)
Accuracy: 0.9904, F1: 0.5061, AUPRC: 0.0017258959896158804
{'enzyme_feature_scaler': MinMaxScaler(), 'reaction_feature_scaler': MinMaxScaler(), 'config': {'layers': [512, 256, 128, 0, 0], 'dropout': 0.05, 'num_epochs': 20, 'batch_size': 1000, 'output_dim': 14, 'learning_rate': 0.0001, 'num_heads': 8, 'early_stop': 5, 'attention': {'product_size': 768, 'substrate_size': 768, 'reaction_size': 2048, 'enzyme_size': 2560, 'embed_size': 1024, 'attention_type': 'cross'}}}
{'enzyme_feature_scaler': MinMaxScaler(), 'reaction_feature_scaler': MinMaxScaler(), 'config': {'layers': [512, 256, 128, 0, 0], 'dropout': 0.05, 'num_epochs': 20, 'batch_size': 1000, 'output_dim': 14, 'learning_rate': 0.0001, 'num_heads': 8, 'early_stop': 5, 'attention': {'product_size': 768, 'substrate_size': 768, 'reaction_size': 2048, 'enzyme_size': 2560, 'embed_size': 1024, 'attention_type': 'cross'}}}
{'enzyme_feature_scaler': MinMaxScaler(), 'reaction_feature_scaler': MinMaxScaler(), 'config': {'lay

/tmp/ipykernel_4159240/1202899340.py:61: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):


ALL (4, 12081, 14)
Accuracy: 0.9998, F1: 0.9210, AUPRC: 0.7111938857158624
{'enzyme_feature_scaler': MinMaxScaler(), 'reaction_feature_scaler': MinMaxScaler(), 'config': {'layers': [512, 256, 128, 0, 0], 'dropout': 0.05, 'num_epochs': 20, 'batch_size': 1000, 'output_dim': 14, 'learning_rate': 0.0001, 'num_heads': 8, 'early_stop': 5, 'attention': {'product_size': 768, 'substrate_size': 768, 'reaction_size': 2048, 'enzyme_size': 2560, 'embed_size': 1024, 'attention_type': 'cross'}}}
{'enzyme_feature_scaler': MinMaxScaler(), 'reaction_feature_scaler': MinMaxScaler(), 'config': {'layers': [512, 256, 128, 0, 0], 'dropout': 0.05, 'num_epochs': 20, 'batch_size': 1000, 'output_dim': 14, 'learning_rate': 0.0001, 'num_heads': 8, 'early_stop': 5, 'attention': {'product_size': 768, 'substrate_size': 768, 'reaction_size': 2048, 'enzyme_size': 2560, 'embed_size': 1024, 'attention_type': 'cross'}}}
{'enzyme_feature_scaler': MinMaxScaler(), 'reaction_feature_scaler': MinMaxScaler(), 'config': {'layers

...

ALL (4, 12081, 14)
Accuracy: 0.9407, F1: 0.4861, AUPRC: 0.001394700139470014
{'enzyme_feature_scaler': MinMaxScaler(), 'reaction_feature_scaler': MinMaxScaler(), 'config': {'layers': [512, 256, 128, 0, 0], 'dropout': 0.05, 'num_epochs': 20, 'batch_size': 1000, 'output_dim': 14, 'learning_rate': 0.0001, 'num_heads': 8, 'early_stop': 5, 'attention': {'product_size': 768, 'substrate_size': 768, 'reaction_size': 2048, 'enzyme_size': 2560, 'embed_size': 1024, 'attention_type': 'cross'}}}
{'enzyme_feature_scaler': MinMaxScaler(), 'reaction_feature_scaler': MinMaxScaler(), 'config': {'layers': [512, 256, 128, 0, 0], 'dropout': 0.05, 'num_epochs': 20, 'batch_size': 1000, 'output_dim': 14, 'learning_rate': 0.0001, 'num_heads': 8, 'early_stop': 5, 'attention': {'product_size': 768, 'substrate_size': 768, 'reaction_size': 2048, 'enzyme_size': 2560, 'embed_size': 1024, 'attention_type': 'cross'}}}
{'enzyme_feature_scaler': MinMaxScaler(), 'reaction_feature_scaler': MinMaxScaler(), 'config': {'laye

/tmp/ipykernel_4159240/1202899340.py:61: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):


ALL (4, 12081, 14)
Accuracy: 0.9863, F1: 0.4966, AUPRC: 0.0009932952570151478
{'enzyme_feature_scaler': MinMaxScaler(), 'reaction_feature_scaler': MinMaxScaler(), 'config': {'layers': [512, 256, 128, 0, 0], 'dropout': 0.05, 'num_epochs': 20, 'batch_size': 1000, 'output_dim': 14, 'learning_rate': 0.0001, 'num_heads': 8, 'early_stop': 5, 'attention': {'product_size': 768, 'substrate_size': 768, 'reaction_size': 2048, 'enzyme_size': 2560, 'embed_size': 1024, 'attention_type': 'cross'}}}
{'enzyme_feature_scaler': MinMaxScaler(), 'reaction_feature_scaler': MinMaxScaler(), 'config': {'layers': [512, 256, 128, 0, 0], 'dropout': 0.05, 'num_epochs': 20, 'batch_size': 1000, 'output_dim': 14, 'learning_rate': 0.0001, 'num_heads': 8, 'early_stop': 5, 'attention': {'product_size': 768, 'substrate_size': 768, 'reaction_size': 2048, 'enzyme_size': 2560, 'embed_size': 1024, 'attention_type': 'cross'}}}
{'enzyme_feature_scaler': MinMaxScaler(), 'reaction_feature_scaler': MinMaxScaler(), 'config': {'lay

/tmp/ipykernel_4159240/1202899340.py:61: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):


RuntimeError: mat1 and mat2 shapes cannot be multiplied (12081x1 and 2048x1024)